In [2]:
import os
import joblib
import pandas as pd
import numpy as np
import shap
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

# -------------------------------
# Set IoT paths
# -------------------------------
THREAT_NAME = "iot"
iot_csv = r"C:\Users\syeds\OneDrive\Documents\Desktop\CyberAI-NextGen-Defender\CyberAI-NextGen-Defender\data\iot\iot.csv"
pkl_model = f"models/{THREAT_NAME}_xgboost_model.pkl"
vectorizer_path = f"models/{THREAT_NAME}_tfidf_vectorizer.pkl"
defense_log = f"models/{THREAT_NAME}_defense_results.csv"
explain_log = f"models/{THREAT_NAME}_explain_log.csv"

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv(iot_csv)

# -------------------------------
# Combine text-like columns for SHAP
# -------------------------------
text_columns = ["sourceID","sourceType","operation","accessedNodeType"]
df["text"] = df[text_columns].astype(str).agg(" ".join, axis=1)

# -------------------------------
# Labels -> normal vs abnormal
# -------------------------------
df["label"] = df["normality"].apply(lambda x: 0 if x=="normal" else 1)

# -------------------------------
# Train XGBoost if PKL not exists
# -------------------------------
if not os.path.exists(pkl_model):
    X_features = df["text"]
    y = df["label"]

    vectorizer = TfidfVectorizer(max_features=3000)
    X_tfidf = vectorizer.fit_transform(X_features)
    
    clf = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss')
    clf.fit(X_tfidf, y)
    
    os.makedirs("models", exist_ok=True)
    joblib.dump(clf, pkl_model)
    joblib.dump(vectorizer, vectorizer_path)
    print(f"Saved PKL model -> {pkl_model} and TF-IDF vectorizer -> {vectorizer_path}")
else:
    print(f"PKL model already exists -> {pkl_model}")
    clf = joblib.load(pkl_model)
    vectorizer = joblib.load(vectorizer_path)

# -------------------------------
# Defense log (simulated)
# -------------------------------
df["defense_action"] = df["label"].apply(lambda x: "quarantine" if x==1 else "allow")
df.to_csv(defense_log, index=False)
print(f"Saved defense results -> {defense_log}")

# -------------------------------
# SHAP Explainability
# -------------------------------
quarantine_samples = df[df["defense_action"]=="quarantine"].copy()
texts = quarantine_samples["text"].tolist()

if len(texts) > 0:
    X_tfidf = vectorizer.transform(texts)
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_tfidf)

    feature_names = np.array(vectorizer.get_feature_names_out())
    explain_data = []

    for i, msg in enumerate(texts):
        shap_row = shap_values[i].flatten()
        top_indices = np.argsort(np.abs(shap_row))[-10:][::-1]
        top_features = feature_names[top_indices]
        top_contribs = shap_row[top_indices]
        top_pairs = [f"{w} ({round(c,3)})" for w,c in zip(top_features, top_contribs)]
        explain_data.append({
            "index": quarantine_samples.index[i],
            "text": msg[:120] + ("..." if len(msg) > 120 else ""),
            "top_contributing_words": ", ".join(top_pairs)
        })

    explain_df = pd.DataFrame(explain_data)
    explain_df.to_csv(explain_log, index=False)
    print(f"Explainability log saved -> {explain_log}")
else:
    print("⚠️ No quarantined samples. Explain log will be empty.")
    pd.DataFrame().to_csv(explain_log, index=False)


C:\Users\syeds\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [02:30:15] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Saved PKL model -> models/iot_xgboost_model.pkl and TF-IDF vectorizer -> models/iot_tfidf_vectorizer.pkl
Saved defense results -> models/iot_defense_results.csv
Explainability log saved -> models/iot_explain_log.csv
